# VRDet: Stage 03: Real-Time Regime Classifier

**Question:** using only information available at the close of each 15-minute bar, can a model tell which regime
(Quiet / Normal / Chaotic) the market is in *right now*, and raise Chaotic alarms early without crying wolf?

- **Target:** HMM hindsight labels from Stage 02 (refitted here, same settings, fitted on 2015–2021).
- **Training data:** from 2017 (India VIX intraday data is invalid before Oct 2016, see Stage 01).
- **Testing:** walk-forward, one year at a time, 2022–2026. Each test year is predicted by models trained only
  on earlier years.

Code lives in `src/vrdet/` (`classifiers.py`, `evaluate.py`); this notebook only orchestrates and visualises.

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/agentsmith1337/VolatilityRegimeDetection.git'
DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1Th_ojXA35EyOru2xNko0Hi9TlfNlwv3J?usp=drive_link'  # Colab only: share link of the Drive folder holding the raw parquet files

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = Path('/content/VolatilityRegimeDetection')
    if REPO.exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch', 'gdown', 'hmmlearn', 'lightgbm'], check=True)
    RAW = Path('/content/vrdet_data/raw')
    if not any(RAW.rglob('*.parquet')):
        import re, gdown
        m = re.search(r'folders/([\w-]+)', DRIVE_FOLDER_URL)
        assert m, 'Paste the Drive folder link into DRIVE_FOLDER_URL (and re-run this cell)'
        RAW.mkdir(parents=True, exist_ok=True)
        gdown.download_folder(id=m.group(1), output=str(RAW), quiet=False)
    assert any(RAW.rglob('*.parquet')), ('No parquet files downloaded: in Drive set the folder to '
                                         'Share -> General access -> Anyone with the link')
    OUT = Path('/content/vrdet_data/processed')
else:
    REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    OUT = REPO / 'data' / 'processed'

sys.path.insert(0, str(REPO / 'src'))
if IN_COLAB:
    import shutil
    shutil.rmtree('/content/vrdet', ignore_errors=True)  # old data folder that shadowed the package
for m in [m for m in sys.modules if m == 'vrdet' or m.startswith('vrdet.')]:
    del sys.modules[m]  # forget any previously imported (stale) vrdet
OUT.mkdir(parents=True, exist_ok=True)
print('repo:', REPO, '| colab:', IN_COLAB)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

from vrdet.data import load_raw
from vrdet.features import build_features, add_lags
from vrdet.labels import HMMLabeler, STATE_NAMES
from vrdet.classifiers import (ThresholdClassifier, TunedChaoticThreshold, logistic_regression, lightgbm,
                               garch_live, walk_forward)
from vrdet.evaluate import compare, regime_metrics

pd.set_option('display.width', 200)
COLORS = {0: 'tab:green', 1: 'gold', 2: 'tab:red'}
LABEL_TRAIN_END = '2021-12-31'   # HMM labeler fitted on data up to here (as in Stage 02)
TRAIN_START = '2017-01-01'       # classifiers train from here (valid India VIX)
TEST_YEARS = range(2022, 2027)

## 1. Features and labels
Same pipeline as Stage 02. Extra classifier inputs: |z| (size of the current deseasonalised move) and lags of a few
fast features, so a model can see the last hour's path. All inputs are causal.

In [ ]:
raw = load_raw()
f = build_features(raw['NIFTY_15m'], raw['INDIAVIX_15m'], {'usdinr': raw['USDINR_1d'], 'brent': raw['BRENT_1d']},
                   nifty_daily=raw['NIFTY_1d'], vix_daily=raw['INDIAVIX_1d'])

hmm_lab = HMMLabeler().fit(f, end=LABEL_TRAIN_END)
y = hmm_lab.hindsight(f)                 # target
hmm_live = hmm_lab.live(f)               # benchmark: the HMM's own real-time (filtered) call
p_hmm = hmm_lab.filtered_proba(f).add_prefix('p_hmm_')

f['abs_z'] = f['z'].abs()
f = add_lags(f, ['abs_z', 'range_ds', 'pk_4', 'vix_chg_1'])
X = f.loc[y.index]
FEATURES = list(X.columns)
X = X.join(p_hmm)
HMM_PROBS = list(p_hmm.columns)
idx = X.index

print(len(FEATURES), 'features |', X.shape[0], 'bars |', 'Chaotic share:', round((y == 2).mean(), 3))

## 2. The contenders

| Model | What it is | Why it is here |
|---|---|---|
| Threshold (GK 2h) | 2-hour Garman–Klass volatility above / below training cut-offs | simplest rule a trader would use |
| GARCH(1,1)-t | conditional volatility from a GARCH model, binned the same way | classic econometric volatility model |
| HMM filter | the labelling HMM run forward in real time (no future data) | the benchmark to beat: same model as the target |
| Logistic regression | linear model on all features | interpretable ML |
| LightGBM | gradient-boosted trees on all features | strong non-linear ML |
| LightGBM + HMM | LightGBM that also sees the HMM filter's probabilities | combines both |

Chaotic is rare (~6% of bars), so the ML models call Chaotic when P(Chaotic) ≥ τ. τ is picked on the **last year of
each training window** (never on test data) to maximise Chaotic F1, then the model is refit on the whole window.

In [ ]:
FITTED = {}   # (model, test year) -> (fitted model, feature columns)

def single(make, cols):
    def fit_predict(train, test, train_end):
        return make().fit(X.loc[train, cols], y[train]).predict(X.loc[test, cols])
    return fit_predict

def tuned(name, make, cols):
    def fit_predict(train, test, train_end):
        m = TunedChaoticThreshold(make).fit(X.loc[train, cols], y[train])
        FITTED[name, int(train_end[:4]) + 1] = (m, cols)
        return m.predict(X.loc[test, cols])
    return fit_predict

models = {
    'Threshold (GK 2h)': single(lambda: ThresholdClassifier('gk_8'), ['gk_8']),
    'GARCH(1,1)-t': lambda train, test, train_end: (
        garch_live(f, TRAIN_START, train_end, y[train]).reindex(idx[test]).fillna(1).astype(int).to_numpy()),
    'HMM filter': lambda train, test, train_end: hmm_live.reindex(idx[test]).to_numpy(),
    'Logistic regression': tuned('Logistic regression', logistic_regression, FEATURES),
    'LightGBM': tuned('LightGBM', lightgbm, FEATURES),
    'LightGBM + HMM': tuned('LightGBM + HMM', lightgbm, FEATURES + HMM_PROBS),
}

## 3. Walk-forward test (2022–2026)

In [ ]:
preds = walk_forward(models, idx, TEST_YEARS, TRAIN_START)

taus = pd.Series({k: m.tau_ for k, (m, _) in FITTED.items()}).unstack()
print('Chaotic threshold tau chosen per training window:')
taus

## 4. Results

How to read the table (all on 2022–2026, never seen in training):

| Column | Meaning | Good |
|---|---|---|
| macro_f1 | overall bar-by-bar accuracy, all 3 regimes weighted equally | higher |
| chaotic_precision | when it says Chaotic, how often it is right | higher |
| chaotic_recall | share of truly Chaotic bars it flags | higher |
| episode_recall | share of Chaotic episodes it flags at least once | higher |
| median_delay_bars | 15-min bars from episode start to the first alarm | lower |
| false_alarms_per_week | Chaotic alarms with no real Chaotic bar in them | lower |

In [ ]:
results = compare(y, preds)
results

In [ ]:
by_year = compare(y, {k: preds[k] for k in ['HMM filter', 'LightGBM', 'LightGBM + HMM']}, by_year=True)
by_year[['macro_f1', 'chaotic_recall', 'episode_recall', 'false_alarms_per_week']].unstack(0)

## 5. The alarm-threshold trade-off
Lower τ: more Chaotic alarms, caught earlier, but more false alarms. This is the "sensitivity" setting the app will
expose. Probabilities below come from the same walk-forward models.

In [ ]:
BEST = 'LightGBM + HMM'
proba = pd.concat([pd.DataFrame(m.predict_proba(X.loc[idx.year == yr, cols]), index=idx[idx.year == yr])
                   for (name, yr), (m, cols) in FITTED.items() if name == BEST])
rows = []
for tau in [0.1, 0.2, 0.3, 0.4, 0.5]:
    p = pd.Series(np.where(proba[2] >= tau, 2, proba[[0, 1]].to_numpy().argmax(axis=1)), index=proba.index)
    rows.append({'tau': tau, **regime_metrics(y, p)})
sweep = pd.DataFrame(rows).set_index('tau').round(3)
display(sweep)

h = regime_metrics(y, preds['HMM filter'])
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep['false_alarms_per_week'], sweep['chaotic_recall'], 'o-', label=BEST)
for tau, r in sweep.iterrows():
    ax.annotate(f'τ={tau}', (r['false_alarms_per_week'], r['chaotic_recall']), textcoords='offset points', xytext=(5, -10))
ax.plot(h['false_alarms_per_week'], h['chaotic_recall'], 's', color='tab:red', label='HMM filter')
ax.set_xlabel('false alarms per week'); ax.set_ylabel('Chaotic recall'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Catching chaos vs crying wolf (2022–2026)')
plt.show()

## 6. Where are the mistakes?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name in zip(axes, ['HMM filter', BEST]):
    t, p = y.align(preds[name], join='inner')
    cm = confusion_matrix(t, p, normalize='true')
    ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', color='white' if cm[i, j] > 0.5 else 'black')
    ax.set_xticks(range(3), STATE_NAMES); ax.set_yticks(range(3), STATE_NAMES)
    ax.set_xlabel('predicted (live)'); ax.set_ylabel('true (hindsight)'); ax.set_title(name)
plt.tight_layout(); plt.show()

## 7. Case studies
Top: NIFTY. Strips: true regime (hindsight) and what each model said live. Green = Quiet, yellow = Normal, red = Chaotic.

In [ ]:
def case(start, end, title):
    c = raw['NIFTY_15m']['close'].loc[start:end]
    strips = {'true (hindsight)': y, 'HMM filter': preds['HMM filter'], BEST: preds[BEST]}
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(14, 4.5), sharex=True, gridspec_kw={'height_ratios': [3, 1.3]})
    x = np.arange(len(c))
    ax.plot(x, c.to_numpy(), color='black', lw=1); ax.set_title(title); ax.set_ylabel('NIFTY')
    for row, (name, s) in enumerate(strips.items()):
        v = s.reindex(c.index)
        ax2.scatter(x, np.full(len(x), row), c=[COLORS.get(k, 'lightgrey') for k in v.fillna(-1).astype(int)],
                    marker='s', s=18)
    ax2.set_yticks(range(len(strips)), list(strips)); ax2.set_ylim(-0.7, len(strips) - 0.3)
    days = pd.Series(x, index=c.index).groupby(c.index.date).first()
    ax2.set_xticks(days.to_numpy()[::max(1, len(days) // 10)],
                   [str(d) for d in days.index[::max(1, len(days) // 10)]], rotation=30)
    plt.tight_layout(); plt.show()

case('2022-02-21', '2022-03-04', 'Russia–Ukraine, Feb 2022')
case('2024-05-29', '2024-06-10', 'General election results, Jun 2024')
case('2025-04-01', '2025-04-15', 'US tariff shock, Apr 2025')

## 8. What drives the model?
Feature importance (total gain) of the final-year model. Expect volatility features and the HMM probabilities at the
top; time-of-day and VIX features should add context.

In [ ]:
m, cols = FITTED[BEST, max(TEST_YEARS)]
imp = pd.Series(m.model_.booster_.feature_importance('gain'), index=cols).sort_values()
imp = imp / imp.sum()
imp.tail(20).plot.barh(figsize=(7, 6), title=f'{BEST}: share of total gain (top 20)')
plt.tight_layout(); plt.show()

## 9. Save
Out-of-sample predictions and the results tables, for the report.

In [ ]:
out = pd.DataFrame(preds).join(y.rename('true'))
out.to_parquet(OUT / 'predictions_15m.parquet')
results.to_csv(OUT / 'results_overall.csv')
by_year.to_csv(OUT / 'results_by_year.csv')
print('saved', OUT / 'predictions_15m.parquet', out.shape)

## Takeaways (fill in after running)
- Which model wins on the table, and is it worth it over the HMM filter?
- How many Chaotic episodes per year are missed, and are they short ones?
- Typical alarm delay in minutes (bars × 15).